In [199]:
!hostname


c-11-01


In [202]:
import sys
import gzip
import glob
import numpy as np
import pandas as pd

import matplotlib
from matplotlib import pyplot as plt


In [203]:
def load_mgatk_output(output_dir, mito_length):
    # assuming mgatk output naming convention
    base_files = [glob.glob(output_dir + "*.{}.txt.gz".format(nt))[0] for nt in "ATCG"]

    base_coverage_dict = dict()
    for i, nt in enumerate("ATCG"):
        cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)

        # gather coverage for forward strand
        fwd_base_df = cur_base_data[[0, 1, 2]].pivot_table(index=1, columns=0)
        fwd_base_df.columns = [
            x[1] for x in fwd_base_df.columns.values
        ]  # flatten weird multiindex after pivot
        fwd_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in fwd_base_df.columns]
        # fwd_base_df[missing_pos] = 0  # fill in missing positions
        all_columns = [x for x in range(1, mito_length + 1)]
        fwd_base_df = fwd_base_df.reindex(columns=all_columns, fill_value=0)
        fwd_base_df = fwd_base_df.fillna(0).sort_index(
            axis=1
        )  # assume all nan are true zeroes

        # gather coverage for forward strand
        rev_base_df = cur_base_data[[0, 1, 3]].pivot_table(index=1, columns=0)
        rev_base_df.columns = [x[1] for x in rev_base_df.columns.values]
        rev_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in rev_base_df.columns]
        # rev_base_df[missing_pos] = 0
        all_columns = [x for x in range(1, mito_length + 1)]
        rev_base_df = rev_base_df.reindex(columns=all_columns, fill_value=0)
        rev_base_df = rev_base_df.fillna(0).sort_index(axis=1)

        # organize base data into a dict
        base_coverage_dict[nt] = (fwd_base_df, rev_base_df)

    return base_coverage_dict


def gather_possible_variants(base_coverage_dict, reference_file):
    # sum across cells and strands for each base and position
    aggregated_genotype = pd.DataFrame(
        np.zeros((4, mito_length)),
        index=list("ATCG"),
        columns=np.arange(1, mito_length + 1),
    )
    for nt in base_coverage_dict:
        # sum across cells for each strand separately
        fwd_base_df, rev_base_df = base_coverage_dict[nt]
        fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

        # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
        masking = ~(
            (fwd_base_sum > 0) & (rev_base_sum > 0)
        )  # True if position not >0 for both strands
        fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

        # sum across strands
        aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum

    # make a reference set of tuples (pos, ref_base)
    ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]
    ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]
    ref_set = set(
        [(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters]
    )
    ref_dict = dict(ref_set)

    # make an observed set of tuples which are nonzero
    non_zero_idx = np.where(aggregated_genotype > 0)
    non_zero_bases = [letters[i] for i in non_zero_idx[0]]
    non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]
    observed_set = list(zip(non_zero_pos, non_zero_bases))
    observed_set = set(
        [x for x in observed_set if x[0] not in ref_N_positions]
    )  # disregard positions in ref with N

    # take difference between observed and reference
    variant_set = observed_set - ref_set
    variants = sorted(
        [(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0]
    )  # (pos, ref base, obs base)

    return variants


In [204]:
# sys.argv[1]
MGATK_OUT_DIR = "/scr1/users/liuc9/mitochondrial/realdata/01-Sci_Immunol_32651212/cromwell-executions/SCMTAH/022e0012-d8fc-4f0b-89d8-86004847b04a/call-call_mt_variants/execution/cell/final/"
sample_prefix = "cell"  # sys.argv[2]
mito_length = 16569  # int(sys.argv[3])  # 16569
low_coverage_threshold = 10  # int(sys.argv[4])  # 10
mito_genome = "MT"  # sys.argv[5]  # chrM


In [35]:
letters = list("ATCG")


In [42]:
# load_mgatk_output

output_dir = MGATK_OUT_DIR
mito_lenght = mito_length

base_files = [glob.glob(output_dir + "*.{}.txt.gz".format(nt))[0] for nt in "ATCG"]

base_coverage_dict = dict()

# for i, nt in enumerate("ATCG"):
#     cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)
i = 0

cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)

In [46]:
cur_base_data

,0,1,2,3
0,341,TTCCTTCTCAAGTGGG-1,0,1
1,350,TTCCTTCTCAAGTGGG-1,0,2
2,351,TTCCTTCTCAAGTGGG-1,0,2
3,352,TTCCTTCTCAAGTGGG-1,0,2
4,357,TTCCTTCTCAAGTGGG-1,0,2
...,...,...,...,...
5813669,12961,TTCCTTCCAGTCAGCC-1,1,0
5813670,12967,TTCCTTCCAGTCAGCC-1,1,0
5813671,12972,TTCCTTCCAGTCAGCC-1,1,0
5813672,12975,TTCCTTCCAGTCAGCC-1,1,0


In [52]:
fwd_base_df = cur_base_data[[0, 1, 2]].pivot_table(index = 1, columns = 0)
fwd_base_df.columns = [x[1] for x in fwd_base_df.columns.values]
fwd_base_df.index.name = None

In [54]:
all_columns = [x for x in range(1, mito_length + 1)]
fwd_base_df = fwd_base_df.reindex(columns=all_columns, fill_value=0)
fwd_base_df = fwd_base_df.fillna(0).sort_index(axis=1)

In [56]:
# gather coverage for forward strand
rev_base_df = cur_base_data[[0, 1, 3]].pivot_table(index=1, columns=0)
rev_base_df.columns = [x[1] for x in rev_base_df.columns.values]
rev_base_df.index.name = None
# missing_pos = [x for x in range(1, mito_length + 1) if x not in rev_base_df.columns]
# rev_base_df[missing_pos] = 0
all_columns = [x for x in range(1, mito_length + 1)]
rev_base_df = rev_base_df.reindex(columns=all_columns, fill_value=0)
rev_base_df = rev_base_df.fillna(0).sort_index(axis=1)

In [57]:
base_coverage_dict[nt] = (fwd_base_df, rev_base_df)

In [58]:
base_coverage_dict

{'G': (                    1      2      3      4      5      6      7      8      \
  AAACCCACAAGAGCTG-1      0    0.0      0      0    0.0      0    0.0      0   
  AAACCCACACAGCCAC-1      0    2.0      0      0    2.0      0    2.0      0   
  AAACGAACAACGATCT-1      0    0.0      0      0    0.0      0    0.0      0   
  AAACGAACACCAATTG-1      0    0.0      0      0    0.0      0    0.0      0   
  AAACGAAGTACCGCGT-1      0    0.0      0      0    0.0      0    0.0      0   
  ...                   ...    ...    ...    ...    ...    ...    ...    ...   
  TTTGGTTAGTGGTCAG-1      0    0.0      0      0    0.0      0    0.0      0   
  TTTGGTTGTACGAAAT-1      0    0.0      0      0    0.0      0    0.0      0   
  TTTGTTGAGCACCAGA-1      0    0.0      0      0    0.0      0    0.0      0   
  TTTGTTGAGGCTTCCG-1      0    0.0      0      0    0.0      0    0.0      0   
  TTTGTTGCATGAAGCG-1      0    0.0      0      0    0.0      0    0.0      0   
  
                      9      10 

In [59]:
base_coverage_dict = load_mgatk_output(MGATK_OUT_DIR, mito_length)


In [63]:
cell_barcodes = base_coverage_dict["A"][0].index


In [64]:
cell_barcodes


Index(['AAACCCACAAGAGCTG-1', 'AAACCCACACAGCCAC-1', 'AAACGAACAACGATCT-1',
       'AAACGAACACCAATTG-1', 'AAACGAAGTACCGCGT-1', 'AAACGAATCAATCTCT-1',
       'AAACGAATCCGTTTCG-1', 'AAACGCTAGTCCTACA-1', 'AAACGCTGTAATCAGA-1',
       'AAAGGATGTATACCCA-1',
       ...
       'TTTGATCCATGAAGCG-1', 'TTTGATCTCAGCCTCT-1', 'TTTGATCTCGTCAGAT-1',
       'TTTGGAGGTGCAACGA-1', 'TTTGGTTAGGCAGTCA-1', 'TTTGGTTAGTGGTCAG-1',
       'TTTGGTTGTACGAAAT-1', 'TTTGTTGAGCACCAGA-1', 'TTTGTTGAGGCTTCCG-1',
       'TTTGTTGCATGAAGCG-1'],
      dtype='object', length=2368)

In [66]:
# total coverage per position per cell

total_coverage = pd.DataFrame(
    np.zeros((len(cell_barcodes), mito_length)),
    index=cell_barcodes,
    columns=np.arange(1, mito_length + 1),
)
total_coverage

,1,2,3,4,5,6,7,8,9,10,...,16560,16561,16562,16563,16564,16565,16566,16567,16568,16569
AAACCCACAAGAGCTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCCACACAGCCAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAACAACGATCT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAACACCAATTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAAGTACCGCGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTAGTGGTCAG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGGTTGTACGAAAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGAGCACCAGA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGAGGCTTCCG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
for nt in base_coverage_dict:
    total_coverage += base_coverage_dict[nt][0]
    total_coverage += base_coverage_dict[nt][1]


In [68]:
total_coverage

,1,2,3,4,5,6,7,8,9,10,...,16560,16561,16562,16563,16564,16565,16566,16567,16568,16569
AAACCCACAAGAGCTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCCACACAGCCAC-1,2.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,4.0,4.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0,3.0
AAACGAACAACGATCT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAACACCAATTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAAGTACCGCGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGTTAGTGGTCAG-1,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
TTTGGTTGTACGAAAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGAGCACCAGA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGAGGCTTCCG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [71]:
total_coverage.mean(axis = 1) 

AAACCCACAAGAGCTG-1     26.099523
AAACCCACACAGCCAC-1    124.033074
AAACGAACAACGATCT-1     12.782546
AAACGAACACCAATTG-1      0.299113
AAACGAAGTACCGCGT-1     59.352767
                         ...    
TTTGGTTAGTGGTCAG-1     31.739936
TTTGGTTGTACGAAAT-1     19.745066
TTTGTTGAGCACCAGA-1     60.985274
TTTGTTGAGGCTTCCG-1     30.957089
TTTGTTGCATGAAGCG-1      0.103989
Length: 2368, dtype: float64

In [72]:
# exclude low coverage cells from variant calling

cell_barcodes = total_coverage.index[
    total_coverage.mean(axis=1) > low_coverage_threshold
]


In [73]:
for nt in base_coverage_dict:
    base_coverage_dict[nt] = (
        base_coverage_dict[nt][0].loc[cell_barcodes, :],
        base_coverage_dict[nt][1].loc[cell_barcodes, :],
    )
total_coverage = total_coverage.loc[cell_barcodes, :]


In [76]:
reference_file = MGATK_OUT_DIR + mito_genome + "_refAllele.txt"

aggregated_genotype = pd.DataFrame(
  np.zeros((4, len(mito_genome))),
  index = list("ATCG"),
  columns = np.arange(1, len(mito_genome) + 1)
)

In [78]:
nt = "A"

In [79]:
fwd_base_df, rev_base_df = base_coverage_dict[nt]
fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

In [90]:
masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))
fwd_base_sum[masking], rev_base_sum[masking] = 0, 0


In [94]:
aggregated_genotype = pd.DataFrame(np.zeros((4, mito_length)), index=list("ATCG"), columns=np.arange(1, mito_length + 1))
for nt in base_coverage_dict:
    # sum across cells for each strand separately
    fwd_base_df, rev_base_df = base_coverage_dict[nt]
    fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

    # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
    masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))  # True if position not >0 for both strands
    fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

    # sum across strands
    aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum

In [95]:
aggregated_genotype

,1,2,3,4,5,6,7,8,9,10,...,16560,16561,16562,16563,16564,16565,16566,16567,16568,16569
A,0.0,564.0,0.0,0.0,706.0,0.0,814.0,0.0,0.0,0.0,...,0.0,2030.0,0.0,0.0,1926.0,0.0,0.0,1843.0,0.0,0.0
T,0.0,0.0,601.0,0.0,0.0,0.0,0.0,0.0,0.0,973.0,...,0.0,0.0,2005.0,0.0,0.0,0.0,0.0,0.0,1796.0,0.0
C,0.0,0.0,0.0,686.0,0.0,789.0,0.0,0.0,0.0,0.0,...,2136.0,3.0,0.0,1984.0,0.0,1910.0,0.0,0.0,0.0,0.0
G,488.0,0.0,0.0,0.0,0.0,0.0,0.0,883.0,923.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1860.0,0.0,0.0,1646.0


In [96]:
ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]


In [99]:
ref_set[:4]

[['1', 'G'], ['2', 'A'], ['3', 'T'], ['4', 'C']]

In [100]:
ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]

In [101]:
ref_N_positions

[3107]

In [102]:

ref_set = set([(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters])
ref_dict = dict(ref_set)


In [106]:
aggregated_genotype

,1,2,3,4,5,6,7,8,9,10,...,16560,16561,16562,16563,16564,16565,16566,16567,16568,16569
A,0.0,564.0,0.0,0.0,706.0,0.0,814.0,0.0,0.0,0.0,...,0.0,2030.0,0.0,0.0,1926.0,0.0,0.0,1843.0,0.0,0.0
T,0.0,0.0,601.0,0.0,0.0,0.0,0.0,0.0,0.0,973.0,...,0.0,0.0,2005.0,0.0,0.0,0.0,0.0,0.0,1796.0,0.0
C,0.0,0.0,0.0,686.0,0.0,789.0,0.0,0.0,0.0,0.0,...,2136.0,3.0,0.0,1984.0,0.0,1910.0,0.0,0.0,0.0,0.0
G,488.0,0.0,0.0,0.0,0.0,0.0,0.0,883.0,923.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1860.0,0.0,0.0,1646.0


In [111]:
a = pd.DataFrame({'a': [1, 0, 3], 'b': [4, 5, 6]})

In [114]:
a

,a,b
0,1,4
1,0,5
2,3,6


In [113]:
np.where(a > 0)

(array([0, 0, 1, 2, 2]), array([0, 1, 1, 0, 1]))

In [119]:
aggregated_genotype

,1,2,3,4,5,6,7,8,9,10,...,16560,16561,16562,16563,16564,16565,16566,16567,16568,16569
A,0.0,564.0,0.0,0.0,706.0,0.0,814.0,0.0,0.0,0.0,...,0.0,2030.0,0.0,0.0,1926.0,0.0,0.0,1843.0,0.0,0.0
T,0.0,0.0,601.0,0.0,0.0,0.0,0.0,0.0,0.0,973.0,...,0.0,0.0,2005.0,0.0,0.0,0.0,0.0,0.0,1796.0,0.0
C,0.0,0.0,0.0,686.0,0.0,789.0,0.0,0.0,0.0,0.0,...,2136.0,3.0,0.0,1984.0,0.0,1910.0,0.0,0.0,0.0,0.0
G,488.0,0.0,0.0,0.0,0.0,0.0,0.0,883.0,923.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1860.0,0.0,0.0,1646.0


In [139]:
non_zero_idx = np.where(aggregated_genotype > 0)


In [140]:
non_zero_bases = [letters[i] for i in non_zero_idx[0]]
non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]

In [141]:
non_zero_bases[:4]

['A', 'A', 'A', 'A']

In [142]:
non_zero_pos[:4]

[2, 5, 7, 13]

In [143]:
observed_set = list(zip(non_zero_pos, non_zero_base))

In [144]:
observed_set[:5]

[(2, 'A'), (5, 'A'), (7, 'A'), (13, 'A'), (16, 'A')]

In [145]:
bserved_set = set([x for x in observed_set if x[0] not in ref_N_positions])  # disregard position

In [151]:
bserved_set[0:10]

TypeError: 'set' object is not subscriptable

In [146]:
list(bserved_set)[0:10]

[(2120, 'T'),
 (8378, 'A'),
 (5798, 'C'),
 (15803, 'G'),
 (14009, 'T'),
 (6205, 'G'),
 (5809, 'G'),
 (5943, 'G'),
 (2808, 'T'),
 (2150, 'T')]

In [147]:
list(ref_set)[0:10]

[(11094, 'C'),
 (5992, 'G'),
 (617, 'G'),
 (221, 'G'),
 (2473, 'A'),
 (8378, 'A'),
 (13887, 'A'),
 (10456, 'A'),
 (4685, 'A'),
 (10328, 'A')]

In [153]:
variant_set = set(observed_set) - set(ref_set)

In [155]:
variant_set

{(11228, 'C'),
 (2120, 'T'),
 (10832, 'C'),
 (13796, 'T'),
 (14283, 'A'),
 (289, 'C'),
 (300, 'G'),
 (11273, 'T'),
 (7976, 'T'),
 (14100, 'A'),
 (13838, 'A'),
 (14951, 'C'),
 (11124, 'C'),
 (11258, 'C'),
 (1888, 'T'),
 (14313, 'A'),
 (14051, 'A'),
 (11733, 'C'),
 (14506, 'C'),
 (11303, 'T'),
 (11075, 'C'),
 (13777, 'T'),
 (11209, 'C'),
 (8621, 'A'),
 (14130, 'A'),
 (13868, 'A'),
 (6269, 'T'),
 (13310, 'G'),
 (11778, 'T'),
 (15388, 'G'),
 (14343, 'A'),
 (3518, 'C'),
 (14536, 'C'),
 (3925, 'G'),
 (15339, 'G'),
 (3663, 'G'),
 (406, 'A'),
 (14294, 'A'),
 (8523, 'A'),
 (3731, 'C'),
 (9468, 'T'),
 (3563, 'T'),
 (14977, 'T'),
 (11546, 'T'),
 (6524, 'A'),
 (3810, 'C'),
 (8044, 'G'),
 (9285, 'T'),
 (14566, 'C'),
 (14324, 'A'),
 (3499, 'C'),
 (13108, 'G'),
 (34, 'T'),
 (387, 'A'),
 (11109, 'G'),
 (14958, 'T'),
 (6018, 'T'),
 (3791, 'C'),
 (9266, 'T'),
 (11831, 'A'),
 (10470, 'C'),
 (11139, 'G'),
 (7173, 'C'),
 (7842, 'G'),
 (13157, 'C'),
 (12499, 'C'),
 (9692, 'T'),
 (3787, 'T'),
 (3921, 'T'),
 

In [156]:
ref_dict[1]

'G'

In [158]:
variants = sorted([(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0]) 

KeyError: 3107

In [110]:
non_zero_idx[1]

array([    1,     4,     6, ..., 16557, 16565, 16568])

In [82]:
masking

1         True
2        False
3         True
4         True
5        False
         ...  
16565     True
16566     True
16567    False
16568     True
16569     True
Length: 16569, dtype: bool

In [75]:
reference_file

'/scr1/users/liuc9/mitochondrial/realdata/01-Sci_Immunol_32651212/cromwell-executions/SCMTAH/022e0012-d8fc-4f0b-89d8-86004847b04a/call-call_mt_variants/execution/cell/final/MT_refAllele.txt'

In [159]:
# cell positional variants
variants = gather_possible_variants(
    base_coverage_dict, MGATK_OUT_DIR + mito_genome + "_refAllele.txt"
)


In [163]:
len(variants)

6231

In [161]:
variant_names = ["{}{}>{}".format(x[0], x[1], x[2]) for x in variants]


In [162]:
variant_names

['34G>T',
 '39C>A',
 '45A>C',
 '46T>G',
 '49A>C',
 '51T>A',
 '52T>G',
 '58T>G',
 '65T>G',
 '73A>G',
 '75G>A',
 '76C>T',
 '76C>A',
 '79G>T',
 '81G>T',
 '82A>G',
 '91C>T',
 '96C>T',
 '96C>A',
 '97G>A',
 '98C>A',
 '106G>A',
 '110C>T',
 '110C>A',
 '117T>G',
 '121G>A',
 '125T>G',
 '134T>G',
 '137A>T',
 '138T>C',
 '145C>A',
 '149T>G',
 '150C>T',
 '154T>G',
 '157T>G',
 '163G>A',
 '169A>C',
 '172T>G',
 '173T>C',
 '175A>C',
 '182C>T',
 '183A>T',
 '183A>G',
 '187G>T',
 '189A>C',
 '195T>G',
 '199T>A',
 '201A>T',
 '215A>T',
 '219A>T',
 '221G>T',
 '222C>A',
 '225G>T',
 '227A>G',
 '229G>A',
 '229G>T',
 '230A>C',
 '237A>G',
 '237A>T',
 '238A>C',
 '239T>G',
 '240A>G',
 '242C>A',
 '248A>G',
 '249A>C',
 '254T>A',
 '257A>G',
 '259A>C',
 '263A>T',
 '263A>G',
 '264C>A',
 '265T>G',
 '266T>C',
 '269C>A',
 '270A>C',
 '272A>C',
 '274A>C',
 '276A>C',
 '278A>C',
 '279T>G',
 '280C>G',
 '282T>G',
 '283A>C',
 '284A>C',
 '286A>C',
 '287A>C',
 '288A>C',
 '289A>C',
 '290A>T',
 '291A>T',
 '292T>A',
 '292T>G',
 '293T>G'

In [164]:
# build two <cell by variant tables>, one for each strand
total_coverage_variant_df = []
fwd_cell_variant_df, rev_cell_variant_df = [], []


In [166]:
total_coverage[2]


AAACCCACAAGAGCTG-1    0.0
AAACCCACACAGCCAC-1    3.0
AAACGAACAACGATCT-1    0.0
AAACGAAGTACCGCGT-1    0.0
AAACGAATCAATCTCT-1    0.0
                     ... 
TTTGGAGGTGCAACGA-1    0.0
TTTGGTTAGTGGTCAG-1    0.0
TTTGGTTGTACGAAAT-1    0.0
TTTGTTGAGCACCAGA-1    0.0
TTTGTTGAGGCTTCCG-1    0.0
Name: 2, Length: 1256, dtype: float64

In [179]:
base_coverage_dict["A"][0][3].values

array([0, 0, 0, ..., 0, 0, 0])

In [167]:
for i, var in enumerate(variants):
    var_name = variant_names[i]
    pos, base = var[0], var[2]
    total_coverage_variant_df.append(total_coverage[pos])
    fwd_cell_variant_df.append(base_coverage_dict[base][0][pos].values)
    rev_cell_variant_df.append(base_coverage_dict[base][1][pos].values)


In [169]:
fwd_cell_variant_df

[array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 6., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 

In [170]:
total_coverage_variant_df = pd.DataFrame(np.array(total_coverage_variant_df).T, index=cell_barcodes, columns=variant_names)


In [171]:
total_coverage_variant_df

,34G>T,39C>A,45A>C,46T>G,49A>C,51T>A,52T>G,58T>G,65T>G,73A>G,...,16534A>C,16534A>G,16535G>T,16537C>A,16538C>A,16542C>T,16543G>T,16547C>A,16558G>T,16561A>C
AAACCCACAAGAGCTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCCACACAGCCAC-1,6.0,8.0,10.0,10.0,11.0,12.0,12.0,13.0,14.0,15.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
AAACGAACAACGATCT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAAGTACCGCGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAATCAATCTCT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGAGGTGCAACGA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGGTTAGTGGTCAG-1,1.0,1.0,1.0,2.0,2.0,2.0,2.0,3.0,3.0,3.0,...,2.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0
TTTGGTTGTACGAAAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGAGCACCAGA-1,0.0,0.0,1.0,1.0,1.0,1.0,1.0,5.0,6.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [172]:
fwd_cell_variant_df = pd.DataFrame(np.array(fwd_cell_variant_df).T, index=cell_barcodes, columns=variant_names)
rev_cell_variant_df = pd.DataFrame(np.array(rev_cell_variant_df).T, index=cell_barcodes, columns=variant_names)


In [173]:
all_cell_variant_df = fwd_cell_variant_df + rev_cell_variant_df

In [174]:
all_cell_variant_df

,34G>T,39C>A,45A>C,46T>G,49A>C,51T>A,52T>G,58T>G,65T>G,73A>G,...,16534A>C,16534A>G,16535G>T,16537C>A,16538C>A,16542C>T,16543G>T,16547C>A,16558G>T,16561A>C
AAACCCACAAGAGCTG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCCACACAGCCAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAACAACGATCT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAAGTACCGCGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAATCAATCTCT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGAGGTGCAACGA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGGTTAGTGGTCAG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGGTTGTACGAAAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGTTGAGCACCAGA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [180]:
heteroplasmic_df = all_cell_variant_df / total_coverage_variant_df

In [181]:
heteroplasmic_df

,34G>T,39C>A,45A>C,46T>G,49A>C,51T>A,52T>G,58T>G,65T>G,73A>G,...,16534A>C,16534A>G,16535G>T,16537C>A,16538C>A,16542C>T,16543G>T,16547C>A,16558G>T,16561A>C
AAACCCACAAGAGCTG-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACCCACACAGCCAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAACAACGATCT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACGAAGTACCGCGT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACGAATCAATCTCT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGAGGTGCAACGA-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TTTGGTTAGTGGTCAG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGGTTGTACGAAAT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TTTGTTGAGCACCAGA-1,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [182]:
mask_idx = (fwd_cell_variant_df + rev_cell_variant_df) == 0  # set 0 on both strands to nan to exclude from correlation calculation
fwd_cell_variant_df[mask_idx] = np.nan
rev_cell_variant_df[mask_idx] = np.nan


In [183]:
fwd_cell_variant_df

,34G>T,39C>A,45A>C,46T>G,49A>C,51T>A,52T>G,58T>G,65T>G,73A>G,...,16534A>C,16534A>G,16535G>T,16537C>A,16538C>A,16542C>T,16543G>T,16547C>A,16558G>T,16561A>C
AAACCCACAAGAGCTG-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACCCACACAGCCAC-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACGAACAACGATCT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACGAAGTACCGCGT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACGAATCAATCTCT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGAGGTGCAACGA-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TTTGGTTAGTGGTCAG-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TTTGGTTGTACGAAAT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TTTGTTGAGCACCAGA-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [184]:
variant_strand_corr = fwd_cell_variant_df.corrwith(rev_cell_variant_df).round(3)

In [185]:
variant_strand_corr

34G>T      -1.0
39C>A      -1.0
45A>C      -1.0
46T>G      -1.0
49A>C      -1.0
           ... 
16542C>T   -1.0
16543G>T   -1.0
16547C>A   -1.0
16558G>T   -1.0
16561A>C   -1.0
Length: 6231, dtype: float64

In [187]:
variant_mean = all_cell_variant_df.sum() / total_coverage_variant_df.sum()
variant_var = heteroplasmic_df.var()


In [188]:
variant_vmr = variant_var / (variant_mean + 0.00000000001)

In [189]:
variant_vmr

34G>T       0.721038
39C>A       0.036336
45A>C       0.105543
46T>G       0.080974
49A>C       0.022477
              ...   
16542C>T    0.078556
16543G>T    0.025245
16547C>A    0.038654
16558G>T    1.481099
16561A>C    2.216960
Length: 6231, dtype: float64

In [190]:
variants

[(34, 'G', 'T'),
 (39, 'C', 'A'),
 (45, 'A', 'C'),
 (46, 'T', 'G'),
 (49, 'A', 'C'),
 (51, 'T', 'A'),
 (52, 'T', 'G'),
 (58, 'T', 'G'),
 (65, 'T', 'G'),
 (73, 'A', 'G'),
 (75, 'G', 'A'),
 (76, 'C', 'T'),
 (76, 'C', 'A'),
 (79, 'G', 'T'),
 (81, 'G', 'T'),
 (82, 'A', 'G'),
 (91, 'C', 'T'),
 (96, 'C', 'T'),
 (96, 'C', 'A'),
 (97, 'G', 'A'),
 (98, 'C', 'A'),
 (106, 'G', 'A'),
 (110, 'C', 'T'),
 (110, 'C', 'A'),
 (117, 'T', 'G'),
 (121, 'G', 'A'),
 (125, 'T', 'G'),
 (134, 'T', 'G'),
 (137, 'A', 'T'),
 (138, 'T', 'C'),
 (145, 'C', 'A'),
 (149, 'T', 'G'),
 (150, 'C', 'T'),
 (154, 'T', 'G'),
 (157, 'T', 'G'),
 (163, 'G', 'A'),
 (169, 'A', 'C'),
 (172, 'T', 'G'),
 (173, 'T', 'C'),
 (175, 'A', 'C'),
 (182, 'C', 'T'),
 (183, 'A', 'T'),
 (183, 'A', 'G'),
 (187, 'G', 'T'),
 (189, 'A', 'C'),
 (195, 'T', 'G'),
 (199, 'T', 'A'),
 (201, 'A', 'T'),
 (215, 'A', 'T'),
 (219, 'A', 'T'),
 (221, 'G', 'T'),
 (222, 'C', 'A'),
 (225, 'G', 'T'),
 (227, 'A', 'G'),
 (229, 'G', 'A'),
 (229, 'G', 'T'),
 (230, 'A', '

In [193]:
((fwd_cell_variant_df >= 2) & (rev_cell_variant_df >= 2)).sum()

34G>T       0
39C>A       0
45A>C       0
46T>G       0
49A>C       0
           ..
16542C>T    0
16543G>T    0
16547C>A    0
16558G>T    0
16561A>C    0
Length: 6231, dtype: int64

In [194]:
heteroplasmic_df

,34G>T,39C>A,45A>C,46T>G,49A>C,51T>A,52T>G,58T>G,65T>G,73A>G,...,16534A>C,16534A>G,16535G>T,16537C>A,16538C>A,16542C>T,16543G>T,16547C>A,16558G>T,16561A>C
AAACCCACAAGAGCTG-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACCCACACAGCCAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGAACAACGATCT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACGAAGTACCGCGT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AAACGAATCAATCTCT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGGAGGTGCAACGA-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TTTGGTTAGTGGTCAG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TTTGGTTGTACGAAAT-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TTTGTTGAGCACCAGA-1,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [195]:
# compute other summary stats
variant_positon = [x[0] for x in variants]
variant_nucleotide = ["{}>{}".format(x[1], x[2]) for x in variants]
variant_n_cells_conf_detected = ((fwd_cell_variant_df >= 2) & (rev_cell_variant_df >= 2)).sum()
variant_n_cells_over_5 = (heteroplasmic_df >= 0.05).sum()
variant_n_cells_over_10 = (heteroplasmic_df >= 0.1).sum()
variant_n_cells_over_20 = (heteroplasmic_df >= 0.2).sum()
variant_n_cells_over_95 = (heteroplasmic_df >= 0.95).sum()
max_heteroplasmy = heteroplasmic_df.max()
variant_mean_coverage = total_coverage_variant_df.mean()

In [196]:
# pack summary stats
variant_output = pd.DataFrame(
    [
        variant_positon,
        variant_nucleotide,
        variant_names,
        variant_vmr,
        variant_mean,
        variant_var,
        variant_n_cells_conf_detected,
        variant_n_cells_over_5,
        variant_n_cells_over_10,
        variant_n_cells_over_20,
        variant_n_cells_over_95,
        max_heteroplasmy,
        variant_strand_corr,
        variant_mean_coverage,
    ]
).T
variant_output.columns = ["position", "nucleotide", "variant", "vmr", "mean", "variance", "n_cells_conf_detected", "n_cells_over_5", "n_cells_over_10", "n_cells_over_20", "n_cells_over_95", "max_heteroplasmy", "strand_correlation", "mean_coverage"]
variant_output[["vmr", "mean", "variance", "strand_correlation", "mean_coverage", "max_heteroplasmy"]] = variant_output[["vmr", "mean", "variance", "strand_correlation", "mean_coverage", "max_heteroplasmy"]].astype(float)


In [197]:
variant_output

,position,nucleotide,variant,vmr,mean,variance,n_cells_conf_detected,n_cells_over_5,n_cells_over_10,n_cells_over_20,n_cells_over_95,max_heteroplasmy,strand_correlation,mean_coverage
0,34,G>T,34G>T,0.721038,0.000871,0.000628,0,2,2,1,0,0.500000,-1.0,1.828025
1,39,C>A,39C>A,0.036336,0.001627,0.000059,0,3,1,0,0,0.111111,-1.0,1.957803
2,45,A>C,45A>C,0.105543,0.001468,0.000155,0,2,1,1,0,0.250000,-1.0,2.168790
3,46,T>G,46T>G,0.080974,0.001420,0.000115,0,4,4,0,0,0.125000,-1.0,2.242834
4,49,A>C,49A>C,0.022477,0.000673,0.000015,0,2,0,0,0,0.062500,-1.0,2.364650
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6226,16542,C>T,16542C>T,0.078556,0.001063,0.000083,0,3,2,0,0,0.142857,-1.0,2.247611
6227,16543,G>T,16543G>T,0.025245,0.000715,0.000018,0,1,0,0,0,0.076923,-1.0,2.226911
6228,16547,C>A,16547C>A,0.038654,0.001116,0.000043,0,2,1,0,0,0.111111,-1.0,2.139331
6229,16558,G>T,16558G>T,1.481099,0.001803,0.002670,0,4,4,1,1,1.000000,-1.0,1.766720
